# 01 — Exploratory Data Analysis

This notebook loads all four raw datasets and performs initial inspection.

**Learning notes:**
- We always inspect dtypes before analysis. Silent type coercion (e.g., a date column read as a string) causes hard-to-spot bugs downstream.
- `date_closed` will have missing values for open/in-progress incidents. This is expected — it is not a data quality problem.
- We save a summary stats CSV at the end. Notebook 05 will read this to build the final report without re-computing.

**By the end of this notebook you will know:**
- The shape and types of every dataset
- Which fields have missing values and why
- How incidents are distributed across teams, categories, severities, and time

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.helpers import load_data, set_plot_style, save_processed

set_plot_style()

In [ ]:
data = load_data("../data/raw")

incidents = data["incidents"]
ris       = data["ris"]
controls  = data["controls"]
mappings  = data["mappings"]

print("Datasets loaded successfully.")

## 1. Shape and Data Types

Before touching the data, we want to know: how many rows and columns does each dataset have, and what type did pandas infer for each column?

In [ ]:
for name, df in data.items():
    print(f"\n--- {name} ---")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(df.dtypes)

## 2. Missing Value Audit

Any column with missing values gets flagged. We expect `date_closed` to have nulls — incidents that are still Open or In Progress have not been closed yet.

In [ ]:
for name, df in data.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f"\n{name} — missing values:")
        print(nulls)
    else:
        print(f"\n{name} — no missing values")

## 3. Sample Rows

Looking at real rows is the fastest way to understand what a dataset contains.

In [ ]:
for name, df in data.items():
    print(f"\n--- {name} (first 3 rows) ---")
    display(df.head(3))

## 4. Incident Distributions

### 4.1 Incidents by Team

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
team_counts = incidents["team"].value_counts()
team_counts.plot(kind="barh", ax=ax)
ax.set_title("Incidents by Team")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig("../data/processed/chart_incidents_by_team.png", dpi=120)
plt.show()
print(team_counts)

### 4.2 Incidents by Category

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
incidents["category"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Incidents by Category")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../data/processed/chart_incidents_by_category.png", dpi=120)
plt.show()

### 4.3 Incidents by Severity

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sev_order = ["P1", "P2", "P3", "P4"]
incidents["severity"].value_counts().reindex(sev_order).plot(kind="bar", ax=ax)
ax.set_title("Incidents by Severity")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../data/processed/chart_incidents_by_severity.png", dpi=120)
plt.show()

### 4.4 Incidents by Status

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
incidents["status"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Incidents by Status")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../data/processed/chart_incidents_by_status.png", dpi=120)
plt.show()

## 5. Remediation Item Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ris["status"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("RIs by Status")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

ris["priority"].value_counts().reindex(["High", "Medium", "Low"]).plot(kind="bar", ax=axes[1])
axes[1].set_title("RIs by Priority")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("../data/processed/chart_ri_distributions.png", dpi=120)
plt.show()

## 6. Incidents Over Time

Grouping by month shows whether incident volume is steady or spiky.

In [ ]:
incidents["month"] = incidents["date_raised"].dt.to_period("M")
monthly = incidents.groupby("month").size().reset_index(name="count")
monthly["month_str"] = monthly["month"].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly["month_str"], monthly["count"])
ax.set_title("Incidents Raised per Month")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../data/processed/chart_incidents_monthly.png", dpi=120)
plt.show()

## 7. Summary Statistics

We save key counts to a CSV so the final report notebook can read them without recomputing.

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Total Incidents",
        "Open Incidents",
        "Total RIs",
        "Open RIs",
        "Total Controls",
        "Total Mappings",
    ],
    "value": [
        len(incidents),
        len(incidents[incidents["status"].isin(["Open", "In Progress"])]),
        len(ris),
        len(ris[ris["status"] != "Closed"]),
        len(controls),
        len(mappings),
    ]
})

path = save_processed(summary, "summary_stats.csv", "../data/processed")
print(f"Saved to: {path}")
display(summary)

## Key Observations

*(Fill these in after running the notebook)*

- **Highest incident team:** 
- **Dominant severity:** 
- **Open incident rate:** 
- **Any monthly spikes?** 

---
**Next:** Run `02_control_mapping.ipynb` to analyse control coverage.